# Post-Training Evaluation: Track A (Clean) vs Track B (Mixed)

This notebook covers **only post-training evaluation** of the RLAD noise-control experiment on Hendrycks MATH (Qwen3-1.7B):

- **Track A (clean):** trained on standard Hendrycks MATH data.
- **Track B (mixed):** trained on Hendrycks MATH with prepended trivia facts.

Both tracks are evaluated at the final **step 400** checkpoint on the **MATH-500** test set using a dedicated 32-sample-per-question eval (`scripts/run_pass32_sweep_1p7_hendrycks.sh`).

It contains two analyses:
1. **Semantic solution diversity** (Section 10) — how varied the 32 solutions per question are, measured with `Qwen/Qwen3-Embedding-4B`.
2. **Pass@k sampling-budget comparison** (Section 11) — unbiased pass@k for `k in {1,2,4,8,16,32}`.

All input artifacts are downloaded from the Modal volume `e3-generation-vol`. We assume the diversity computation (`modal_compute_diversity_hendrycks.py`) has already been run and its `embedding_diversity_results.csv` exists on the volume.

In [ ]:
import os
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## 10. Semantic Solution Diversity (Qwen3-Embedding-4B)

Beyond accuracy, we want to know whether each track explores *diverse reasoning paths* or converges to a narrow set of strategies. We measure this by computing the semantic similarity among all 32 generated solutions for a given question within each track.

**Method:**
1. For each of 100 randomly sampled MATH-500 test questions, extract all 32 response strings from the pass@32 eval parquets (step 400).
2. Encode each response into a normalized embedding vector using `Qwen/Qwen3-Embedding-4B`.
3. Compute pairwise cosine similarity between all `(32 × 31) / 2 = 496` unique pairs of embeddings per question.
4. Average those 496 similarities to get one "mean intra-question similarity" score per question.
5. Compare the distribution of these scores between Track A and Track B.

**Interpretation:** A *higher* mean similarity means the 32 solutions are semantically very similar to each other (low diversity — the model has settled on a few reasoning templates). A *lower* mean similarity means the solutions vary more in their reasoning (high diversity — the model explores more varied strategies).

**Scope:** We compute similarity *within* each track only (Track A's solutions compared to Track A's, Track B's compared to Track B's). The same 100 questions are used for both tracks to ensure a fair comparison. All 32 responses per question are used, regardless of correctness.

The underlying computation runs on a Modal GPU instance (`modal_compute_diversity_hendrycks.py`); here we assume it has already run and simply download the resulting CSV.

In [ ]:
# Download the diversity results from the Modal volume (skip if present).
DIVERSITY_DIR = "diversity_1p7b_hendrycks"
os.makedirs(DIVERSITY_DIR, exist_ok=True)

remote_csv = "diversity_1p7b_hendrycks/embedding_diversity_results.csv"
local_csv = os.path.join(DIVERSITY_DIR, "embedding_diversity_results.csv")

if not os.path.exists(local_csv):
    print(f"Downloading {remote_csv} from Modal volume 'e3-generation-vol'...")
    result = subprocess.run(
        ["modal", "volume", "get", "e3-generation-vol", remote_csv],
        capture_output=True, text=True, cwd=DIVERSITY_DIR,
    )
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("Failed to download diversity results")
    print("Download complete.")
else:
    print(f"{local_csv} already present locally.")

assert os.path.exists(local_csv), f"Missing: {local_csv}"
print(f"OK: {local_csv} ({os.path.getsize(local_csv)} bytes)")

In [ ]:
# Load the diversity results
df_div = pd.read_csv(os.path.join("diversity_1p7b_hendrycks", "embedding_diversity_results.csv"))

div_a = df_div[df_div["track"] == "A"]["mean_similarity"]
div_b = df_div[df_div["track"] == "B"]["mean_similarity"]

print("=== Mean Intra-Question Similarity (lower = more diverse) ===")
print(f"Track A (Clean): mean={div_a.mean():.4f}  std={div_a.std():.4f}  "
      f"min={div_a.min():.4f}  max={div_a.max():.4f}  n={len(div_a)}")
print(f"Track B (Mixed): mean={div_b.mean():.4f}  std={div_b.std():.4f}  "
      f"min={div_b.min():.4f}  max={div_b.max():.4f}  n={len(div_b)}")

# Welch's t-test (does not assume equal variance)
t_stat, p_val = stats.ttest_ind(div_a, div_b, equal_var=False)
print(f"\nWelch's t-test: t={t_stat:.4f}, p={p_val:.4e}")
if p_val < 0.05:
    more_diverse = "Track A" if div_a.mean() < div_b.mean() else "Track B"
    print(f"Statistically significant difference (p<0.05). {more_diverse} has MORE diverse solutions.")
else:
    print("No statistically significant difference in solution diversity (p>=0.05).")

# Plot overlapping density histograms
fig, ax = plt.subplots(figsize=(10, 6))
bins = np.linspace(
    min(div_a.min(), div_b.min()),
    max(div_a.max(), div_b.max()),
    30,
)
ax.hist(div_a, bins=bins, alpha=0.5, color="red", density=True, label="Track A (Clean)")
ax.hist(div_b, bins=bins, alpha=0.5, color="green", density=True, label="Track B (Mixed)")

# Density curves (KDE)
for data, color in [(div_a, "darkred"), (div_b, "darkgreen")]:
    kde = stats.gaussian_kde(data)
    xs = np.linspace(bins[0], bins[-1], 200)
    ax.plot(xs, kde(xs), color=color, linewidth=2)

# Mean lines
ax.axvline(div_a.mean(), color="red", linestyle="--", linewidth=1.5,
           label=f"Track A mean = {div_a.mean():.3f}")
ax.axvline(div_b.mean(), color="green", linestyle="--", linewidth=1.5,
           label=f"Track B mean = {div_b.mean():.3f}")

ax.set_xlabel("Mean Pairwise Cosine Similarity (per question)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Semantic Solution Diversity: Track A vs Track B (Hendrycks MATH, step 400)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("1p7b_hendrycks_embedding_diversity.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved figure -> 1p7b_hendrycks_embedding_diversity.png")

In [ ]:
# Divergence spot-check: which questions show the biggest gap in diversity
# between the two tracks?
pivot = df_div.pivot(index="question_idx", columns="track", values="mean_similarity")
pivot = pivot.dropna()
pivot.columns = ["sim_a", "sim_b"]

# delta = sim_b - sim_a.
#   delta < 0  => Track B more similar (less diverse) than Track A here
#   delta > 0  => Track A more similar (less diverse) than Track B here
pivot["delta"] = pivot["sim_b"] - pivot["sim_a"]

print("=== Questions where Track B is MOST diverse relative to Track A ===")
print("(largest negative delta: Track B solutions much less similar to each other)")
top_b_diverse = pivot.sort_values("delta").head(3)
print(top_b_diverse.to_string())

print("\n=== Questions where Track A is MOST diverse relative to Track B ===")
print("(largest positive delta: Track A solutions much less similar to each other)")
top_a_diverse = pivot.sort_values("delta", ascending=False).head(3)
print(top_a_diverse.to_string())

## 11. Pass@k Sampling-Budget Comparison (up to pass@32)

To study how each track scales with a larger sampling budget, we ran a dedicated evaluation that generates **32 samples per question** on the full MATH-500 test set for the final (step 400) checkpoint of both tracks (see `scripts/run_pass32_sweep_1p7_hendrycks.sh`).

Each eval writes a `per_problem_*.csv` (with `n_correct` out of `n_samples=32` per question). From those counts we compute the **unbiased pass@k estimator** (Chen et al., HumanEval) for `k in {1, 2, 4, 8, 16, 32}` and compare Track A (clean) vs Track B (mixed).

Only the **step 400** checkpoint was evaluated with 32 samples, so this is a single-checkpoint comparison.

In [ ]:
# Download the 32-sample per-problem eval CSVs from the Modal volume (skip if present).
# These are produced by scripts/run_pass32_sweep_1p7_hendrycks.sh and live under
# /data/pass32_1p7b_hendrycks/ on volume 'e3-generation-vol'.
PASS_DIR = "pass32_1p7b_hendrycks"   # local cache
PASS_STEPS = [400]
KS = [1, 2, 4, 8, 16, 32]
os.makedirs(PASS_DIR, exist_ok=True)


def remote_csv_name(track, step):
    return f"per_problem_math_{track}_hendrycks_step{step}.csv"


missing = []
for track in ("track_a", "track_b"):
    for step in PASS_STEPS:
        fname = remote_csv_name(track, step)
        local_path = os.path.join(PASS_DIR, fname)
        if not os.path.exists(local_path):
            missing.append((fname, local_path))

if missing:
    print(f"Downloading {len(missing)} eval CSVs from Modal volume 'e3-generation-vol'...")
    for fname, local_path in missing:
        remote_path = f"pass32_1p7b_hendrycks/{fname}"
        cmd = ["modal", "volume", "get", "e3-generation-vol", remote_path]
        print(f"  {remote_path} -> {local_path}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=PASS_DIR)
        if result.returncode != 0:
            print(f"  ERROR: {result.stderr.strip()}")
            raise RuntimeError(f"Failed to download {remote_path}. Has the sweep finished?")
    print("Download complete.")
else:
    print("All pass@k eval CSVs already present locally.")

# Verify presence
for track in ("track_a", "track_b"):
    for step in PASS_STEPS:
        p = os.path.join(PASS_DIR, remote_csv_name(track, step))
        assert os.path.exists(p), f"Missing: {p}"
print("All pass@k eval CSVs ready.")

In [ ]:
# Compute the unbiased pass@k for k in {1,2,4,8,16,32} from per-problem n_correct.
def pass_at_k(n, c, k):
    """Unbiased pass@k estimator (Chen et al., HumanEval).

    n = samples per question, c = number correct, k = budget.
    Probability that at least one of k drawn samples is correct.
    """
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))


rows = []
for track_label, track_key in (("A", "track_a"), ("B", "track_b")):
    for step in PASS_STEPS:
        df = pd.read_csv(os.path.join(PASS_DIR, remote_csv_name(track_key, step)))
        n_samples = int(df["n_samples"].iloc[0])
        num_problems = len(df)
        for k in KS:
            vals = [pass_at_k(int(r.n_samples), int(r.n_correct), k) for r in df.itertuples()]
            rows.append({
                "track": track_label,
                "step": step,
                "k": k,
                "pass_at_k": float(np.mean(vals)),
                "n_samples": n_samples,
                "num_problems": num_problems,
            })

df_passk = pd.DataFrame(rows)

# Sanity check: pass@k must be non-decreasing in k for each (track, step).
for (t, s), g in df_passk.groupby(["track", "step"]):
    g = g.sort_values("k")
    assert (g["pass_at_k"].diff().dropna() >= -1e-9).all(), f"pass@k not monotonic for track {t} step {s}"

print(f"Computed pass@k for {df_passk[['track','step']].drop_duplicates().shape[0]} (track, step) pairs.")
print(f"Samples per question: {sorted(df_passk['n_samples'].unique())}, "
      f"problems: {sorted(df_passk['num_problems'].unique())}")
df_passk

In [ ]:
# Pass@k curves at step 400: Track A vs Track B.
colors = {"A": "#d62728", "B": "#2ca02c"}
step = 400

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: line plot of pass@k vs k
ax = axes[0]
for label in ("A", "B"):
    d = df_passk[(df_passk["track"] == label) & (df_passk["step"] == step)].sort_values("k")
    ax.plot(d["k"], d["pass_at_k"] * 100, "o-", color=colors[label],
            linewidth=2, markersize=7, label=f"Track {label}")
ax.set_xscale("log", base=2)
ax.set_xticks(KS)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel("k (samples)")
ax.set_ylabel("pass@k (%)")
ax.set_title(f"Pass@k Curve (MATH-500, step {step})")
ax.grid(True, alpha=0.3)
ax.legend()

# Right: grouped bar chart of pass@k by k
ax = axes[1]
x = np.arange(len(KS))
width = 0.38
for i, label in enumerate(("A", "B")):
    d = df_passk[(df_passk["track"] == label) & (df_passk["step"] == step)].sort_values("k")
    ax.bar(x + (i - 0.5) * width, d["pass_at_k"].values * 100, width,
           color=colors[label], alpha=0.85, label=f"Track {label}")
ax.set_xticks(x)
ax.set_xticklabels(KS)
ax.set_xlabel("k (samples)")
ax.set_ylabel("pass@k (%)")
ax.set_title(f"Pass@k by Budget (MATH-500, step {step})")
ax.grid(True, alpha=0.3, axis="y")
ax.legend()

plt.suptitle("Pass@k Sampling-Budget Comparison: Track A vs Track B (Hendrycks MATH)",
             fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("1p7b_hendrycks_pass_at_k.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Summary table: pass@k columns x track rows, plus A-vs-B gap at pass@32.
pivot = (
    df_passk
    .pivot_table(index=["step", "track"], columns="k", values="pass_at_k")
    .mul(100)
    .round(2)
)
pivot.columns = [f"pass@{k}" for k in pivot.columns]

pivot.to_csv("1p7b_hendrycks_pass_at_k.csv")
print("Saved pass@k table to 1p7b_hendrycks_pass_at_k.csv\n")
print(pivot)

# Gap at pass@32 (Track B - Track A) in percentage points.
gap = (
    df_passk[df_passk["k"] == 32]
    .pivot_table(index="step", columns="track", values="pass_at_k")
    .mul(100)
)
gap["gap_B_minus_A"] = (gap["B"] - gap["A"]).round(2)
print("\npass@32 gap (Track B - Track A), percentage points:")
print(gap["gap_B_minus_A"])

## 12. Per-Question Pass@1 Spot-Check (step 400)

Sections 10 and 11 are aggregate. Here we inspect *individual* MATH-500 questions to qualitatively compare how Track A (clean) and Track B (mixed) answer, in the spirit of Section 8 of the original notebook.

We reuse the **same step-400 pass@32 outputs** that back Section 10 (`math_track_{a,b}_hendrycks_step400_outputs.parquet`). For each question we define **pass@1** as the **first of the 32 generated samples** (`responses[0]`), scored with the verl MATH scorer (`\boxed{}` extraction + SymPy-style equivalence). The answer we display is exactly the sample we score.

Three spot-checks (each picks a *random* qualifying question; re-run a cell to draw another):
1. Track B pass@1 **correct**, Track A pass@1 **incorrect**.
2. Track A pass@1 **correct**, Track B pass@1 **incorrect**.
3. **Both** tracks pass@1 **incorrect**.

In [ ]:
# Download the step-400 pass@32 OUTPUT PARQUETS (skip if present).
# These are the same artifacts Section 10's diversity was computed from and
# live under /data/pass32_1p7b_hendrycks/ on volume 'e3-generation-vol'.
# They hold the prompt, ground truth, and all 32 response strings per question.
OUT_DIR = PASS_DIR  # reuse the Section 11 local cache dir


def remote_parquet_name(track, step):
    return f"math_{track}_hendrycks_step{step}_outputs.parquet"


missing = []
for track in ("track_a", "track_b"):
    for step in PASS_STEPS:
        fname = remote_parquet_name(track, step)
        local_path = os.path.join(OUT_DIR, fname)
        if not os.path.exists(local_path):
            missing.append((fname, local_path))

if missing:
    print(f"Downloading {len(missing)} output parquets from Modal volume 'e3-generation-vol'...")
    for fname, local_path in missing:
        remote_path = f"pass32_1p7b_hendrycks/{fname}"
        cmd = ["modal", "volume", "get", "e3-generation-vol", remote_path]
        print(f"  {remote_path} -> {local_path}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=OUT_DIR)
        if result.returncode != 0:
            print(f"  ERROR: {result.stderr.strip()}")
            raise RuntimeError(f"Failed to download {remote_path}.")
    print("Download complete.")
else:
    print("All output parquets already present locally.")

for track in ("track_a", "track_b"):
    for step in PASS_STEPS:
        p = os.path.join(OUT_DIR, remote_parquet_name(track, step))
        assert os.path.exists(p), f"Missing: {p}"
print("All output parquets ready.")

In [ ]:
# Load the step-400 output parquets and score the FIRST sample (pass@1) per question.
import importlib.util
import random
from IPython.display import display, Markdown

STEP = 400
INSTRUCTION = "Let's think step by step and output the final answer within \\boxed{}."

# Import the verl MATH scorer directly from its file. We bypass `import verl`
# because the package __init__ pulls in torch (not needed for pure-string scoring).
def _load_math_scorer():
    here = os.path.abspath(os.getcwd())
    cur = here
    for _ in range(6):
        cand = os.path.join(cur, "verl", "utils", "reward_score", "math.py")
        if os.path.exists(cand):
            spec = importlib.util.spec_from_file_location("verl_math_scorer", cand)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod
        cur = os.path.dirname(cur)
    raise FileNotFoundError("Could not locate verl/utils/reward_score/math.py")

_scorer = _load_math_scorer()


def pass1_correct(response, ground_truth):
    """True if the first sampled response is correct under the MATH scorer."""
    try:
        return _scorer.compute_score(solution_str=response, ground_truth=str(ground_truth)) == 1.0
    except Exception:
        return False


def _question_text(prompt):
    """Extract the user question from the stored prompt, dropping the instruction."""
    content = prompt[0]["content"] if len(prompt) else ""
    return content.replace(INSTRUCTION, "").strip()


df_a_out = pd.read_parquet(os.path.join(PASS_DIR, remote_parquet_name("track_a", STEP)))
df_b_out = pd.read_parquet(os.path.join(PASS_DIR, remote_parquet_name("track_b", STEP)))
assert len(df_a_out) == len(df_b_out), "Track A/B parquets have different #problems"
N = len(df_a_out)

questions = [_question_text(df_a_out.iloc[i]["prompt"]) for i in range(N)]
ground_truth = [str(df_a_out.iloc[i]["reward_model"]["ground_truth"]) for i in range(N)]
resp_a = [df_a_out.iloc[i]["responses"][0] for i in range(N)]
resp_b = [df_b_out.iloc[i]["responses"][0] for i in range(N)]

correct_a = np.array([pass1_correct(resp_a[i], ground_truth[i]) for i in range(N)])
correct_b = np.array([pass1_correct(resp_b[i], ground_truth[i]) for i in range(N)])

print(f"Scored pass@1 (sample 0) for {N} questions at step {STEP}.")
print(f"  Track A pass@1 accuracy: {correct_a.mean()*100:.1f}%  ({correct_a.sum()}/{N})")
print(f"  Track B pass@1 accuracy: {correct_b.mean()*100:.1f}%  ({correct_b.sum()}/{N})")
print(f"  B right / A wrong : {int((correct_b & ~correct_a).sum())}")
print(f"  A right / B wrong : {int((correct_a & ~correct_b).sum())}")
print(f"  Both wrong        : {int((~correct_a & ~correct_b).sum())}")


def _esc(s):
    # Avoid accidental LaTeX/Markdown rendering of '$' so traces show verbatim.
    return str(s).replace("$", "\\$")


def show_example(idx, title):
    ca = bool(correct_a[idx])
    cb = bool(correct_b[idx])
    display(Markdown(f"### {title} — problem_idx = {idx}"))
    display(Markdown(f"**Question**\n\n{_esc(questions[idx])}"))
    display(Markdown(f"**Ground truth:** `{ground_truth[idx]}`"))
    display(Markdown("---"))
    a_tag = "CORRECT" if ca else "INCORRECT"
    b_tag = "CORRECT" if cb else "INCORRECT"
    display(Markdown(f"**Track A (Clean) — pass@1 {a_tag}**  ·  {len(resp_a[idx])} chars"))
    display(Markdown(_esc(resp_a[idx])))
    display(Markdown("---"))
    display(Markdown(f"**Track B (Mixed) — pass@1 {b_tag}**  ·  {len(resp_b[idx])} chars"))
    display(Markdown(_esc(resp_b[idx])))


def pick(mask, label):
    idxs = np.where(mask)[0]
    if len(idxs) == 0:
        print(f"No questions match: {label}")
        return None
    sel = int(random.choice(idxs))
    print(f"{len(idxs)} questions match '{label}'. Showing random idx={sel}.")
    return sel

In [ ]:
# (1) Track B pass@1 correct, Track A pass@1 incorrect.
sel = pick(correct_b & ~correct_a, "Track B right, Track A wrong")
if sel is not None:
    show_example(sel, "Track B right, Track A wrong")

In [ ]:
# (2) Track A pass@1 correct, Track B pass@1 incorrect.
sel = pick(correct_a & ~correct_b, "Track A right, Track B wrong")
if sel is not None:
    show_example(sel, "Track A right, Track B wrong")

In [ ]:
# (3) Both tracks pass@1 incorrect.
sel = pick(~correct_a & ~correct_b, "Both tracks wrong")
if sel is not None:
    show_example(sel, "Both tracks wrong")